# Streambased Core Demo — Hot / Cold Lifecycle

This walkthrough takes you through the lifecycle of data in Streambased, from kafka topic → seamless merged view → cold archive. Each step shows how the population shifts between tiers via a visualization.

In [ ]:
import pandas as pd
import json
import subprocess
import time
import os
from IPython.display import HTML, display
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
def q(sql):
    return spark.sql(sql).toPandas()

In [ ]:
def compose(*args, capture=False):
    host_dir = os.environ["BREAKSTREAM_HOST_DIR"]
    cmd = [
        "docker", "compose",
        "--project-directory", f"{host_dir}/environment",
        "-f", f"{host_dir}/environment/docker-compose.yaml",
    ] + list(args)
    print(">", " ".join(cmd))
    result = subprocess.run(cmd, check=True, capture_output=capture, text=True)
    if capture:
        return result.stdout

In [ ]:
def show_distribution(table='transactions'):
    hot_row = q(f"SELECT COUNT(*) AS cnt, MIN(kafka_offset) AS earliest, MAX(kafka_offset) AS latest FROM isk.hotset.{table}").iloc[0]
    cold_row = q(f"SELECT COUNT(*) AS cnt, MIN(kafka_offset) AS earliest, MAX(kafka_offset) AS latest FROM direct.coldset.{table}").iloc[0]
    merged_row = q(f"SELECT COUNT(*) AS cnt FROM isk.merged.{table}").iloc[0]

    hot_count = int(hot_row['cnt'] or 0)
    cold_count = int(cold_row['cnt'] or 0)
    merged_count = int(merged_row['cnt'] or 0)

    hot_earliest = int(hot_row['earliest']) if hot_row['earliest'] is not None else None
    hot_latest = int(hot_row['latest']) if hot_row['latest'] is not None else None
    cold_earliest = int(cold_row['earliest']) if cold_row['earliest'] is not None else None
    cold_latest = int(cold_row['latest']) if cold_row['latest'] is not None else None

    def fmt_offset(v):
        return str(v) if v is not None else '\u2014'

    denom = max(merged_count, 1)
    hot_pct = round(100 * hot_count / denom)
    cold_pct = round(100 * cold_count / denom)

    total_pct = hot_pct + cold_pct
    if total_pct == 0:
        hot_bar_w = 0
        cold_bar_w = 0
        empty_bar = True
    else:
        hot_bar_w = hot_pct / total_pct * 100
        cold_bar_w = cold_pct / total_pct * 100
        empty_bar = False

    def bar_label(label, width):
        if width > 15:
            return f'<span style="color:#fff;font-size:12px;font-weight:600;pointer-events:none">{label}</span>'
        return ''

    if empty_bar:
        bar_html = '<div style="width:100%;height:32px;background:#e9ecef;border-radius:4px;"></div>'
    else:
        bar_html = f'''
        <div style="display:flex;width:100%;height:32px;border-radius:4px;overflow:hidden;">
          <div style="width:{hot_bar_w:.1f}%;background:#e63946;display:flex;align-items:center;justify-content:center;">
            {bar_label('Hot Set', hot_bar_w)}
          </div>
          <div style="width:{cold_bar_w:.1f}%;background:#4cc9f0;display:flex;align-items:center;justify-content:center;">
            {bar_label('Cold Set', cold_bar_w)}
          </div>
        </div>'''

    def svg_ring(pct, color):
        r = 40
        cx = cy = 50
        circumference = 2 * 3.14159 * r
        dash = circumference * min(pct, 100) / 100
        gap = circumference - dash
        return f'''
        <svg width="100" height="100" viewBox="0 0 100 100">
          <circle cx="{cx}" cy="{cy}" r="{r}" fill="none" stroke="#e9ecef" stroke-width="8"/>
          <circle cx="{cx}" cy="{cy}" r="{r}" fill="none" stroke="{color}" stroke-width="8"
            stroke-dasharray="{dash:.2f} {gap:.2f}"
            stroke-linecap="round"
            transform="rotate(-90 {cx} {cy})"/>
          <text x="{cx}" y="{cy}" text-anchor="middle" dominant-baseline="central"
            style="font-size:16px;font-weight:700;fill:#222;">{pct}%</text>
        </svg>'''

    html = f'''
    <div style="font-family:system-ui,sans-serif;border:1.5px solid #ff6b35;border-radius:8px;padding:16px;max-width:680px;">
      <div style="margin-bottom:8px;">
        <span style="color:#ff6b35;font-weight:700;font-size:15px;">Merged Set</span>
        <span style="color:#222;font-size:15px;margin-left:16px;">Total Rows: <strong>{merged_count:,}</strong></span>
      </div>
      {bar_html}
      <div style="display:flex;gap:16px;margin-top:16px;">
        <div style="flex:1;border:1.5px solid #e63946;border-radius:6px;padding:12px;">
          <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:8px;">
            <span style="color:#e63946;font-weight:700;font-size:14px;">Hot Set</span>
            <span style="color:#999;font-size:12px;">kafka</span>
          </div>
          <div style="display:flex;justify-content:center;margin:8px 0;">
            {svg_ring(hot_pct, '#e63946')}
          </div>
          <table style="width:100%;font-size:13px;border-collapse:collapse;">
            <tr><td style="color:#555;">Total Messages</td><td style="text-align:right;font-weight:600;">{hot_count:,}</td></tr>
            <tr><td style="color:#555;">Earliest Offset</td><td style="text-align:right;">{fmt_offset(hot_earliest)}</td></tr>
            <tr><td style="color:#555;">Latest Offset</td><td style="text-align:right;">{fmt_offset(hot_latest)}</td></tr>
          </table>
        </div>
        <div style="flex:1;border:1.5px solid #4cc9f0;border-radius:6px;padding:12px;">
          <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:8px;">
            <span style="color:#4cc9f0;font-weight:700;font-size:14px;">Cold Set</span>
            <span style="color:#999;font-size:12px;">iceberg</span>
          </div>
          <div style="display:flex;justify-content:center;margin:8px 0;">
            {svg_ring(cold_pct, '#4cc9f0')}
          </div>
          <table style="width:100%;font-size:13px;border-collapse:collapse;">
            <tr><td style="color:#555;">Total Rows</td><td style="text-align:right;font-weight:600;">{cold_count:,}</td></tr>
            <tr><td style="color:#555;">Earliest Offset</td><td style="text-align:right;">{fmt_offset(cold_earliest)}</td></tr>
            <tr><td style="color:#555;">Latest Offset</td><td style="text-align:right;">{fmt_offset(cold_latest)}</td></tr>
          </table>
        </div>
      </div>
    </div>
    '''
    display(HTML(html))

## Initial state

Before we start traffic, here's what's already in place. The setup phase populated the coldset with seed data; no kafka traffic is flowing yet.

We begin by describing the databases available. We see:

* **hotset** — data from Kafka only
* **coldset** — data from Iceberg only
* **merged** — a hotset and coldset seamlessly unioned by Streambased.

Only the coldset database physically exists, hotset and merged are virtual databases provided by Streambased I.S.K.

In [ ]:
q("SHOW DATABASES IN isk")

Let's list hotset tables. These correspond exactly to topics in Kafka.

If new topics are added they will immediately appear here.

The accounts table exists in Kafka only and has no Iceberg equivalent.

In [ ]:
q("SHOW TABLES IN isk.hotset")

Now let's list coldset tables. These correspond exactly to tables in Iceberg.

Again, if new tables are added they will immediately appear here.

The branches table exists in Iceberg only and has no Kafka equivalent.

In [ ]:
q("SHOW TABLES IN direct.coldset")

Finally, let's list merged tables. This is the union of topics in Kafka and tables in Iceberg.

Any that exist in both Kafka and Iceberg (e.g. transactions) are intelligently unioned by Streambased.

In [ ]:
q("SHOW TABLES IN isk.merged")

### Where does the data live right now?

In [ ]:
show_distribution()

## Start the background traffic generator

We now produce continuous events into the kafka topic. Watch the hotset fill up while the coldset stays put.

In [ ]:
compose('up', '-d', 'shadowtraffic_background')
time.sleep(8)

To show the union, let's confirm the data size of the transactions topic on each view.

First the coldset, here we see the transactions data stored in Iceberg.

In [ ]:
q("SELECT COUNT(*) AS coldset_count FROM direct.coldset.transactions")

Now the data size of the transactions topic in the hotset.

Here we see the transactions data stored in Kafka.

In [ ]:
q("SELECT COUNT(*) AS hotset_count FROM isk.hotset.transactions")

And now the data size of the transactions topic in the merged set.

This is a seamless combination of Kafka and Iceberg data.

Why doesn't the merged size equal hotset + coldset?

Streambased guarantees that any queries will run against the latest data available at the time of query execution. As the hotset is being constantly written to, more data is available in it now than there was when we queried the hotset individually.

In [ ]:
q("SELECT COUNT(*) AS merged_count FROM isk.merged.transactions")

### Distribution after traffic has been flowing for a few seconds

In [ ]:
show_distribution()

## Pause traffic to examine schemas cleanly

Now let's look at schema evolution — for example addition of a new field. Note that background data injection is stopped now — to surface messages easier and inject data with evolved schema.

Let's describe the transactions table on the Hotset projection.

In [ ]:
compose('down', 'shadowtraffic_background')
time.sleep(3)

In [ ]:
q("DESCRIBE isk.hotset.transactions")

And fetch last few rows as an example.

Now — let's produce some new messages with additional field FraudRiskScore.

In [ ]:
q("SELECT * FROM isk.hotset.transactions ORDER BY kafka_offset DESC LIMIT 10")

## Inject events with evolved schema

We now run a one-shot setup that adds a new column `FraudRiskScore` to the transactions topic, then start a new background generator that emits events with this evolved schema.

In [ ]:
compose('up', '--wait', 'shadowtraffic_setup_evolved')

In [ ]:
compose('up', '-d', 'shadowtraffic_background_evolved')
time.sleep(8)

Now that new data is coming into the topic with evolved schema (FraudRiskScore optional field added) — let's describe the table again.

In [ ]:
q("DESCRIBE isk.hotset.transactions")

And check the latest events added to it. Note the newest events have FraudRiskScore populated — while older ones are rendered with null (events pre-schema change).

Now we can restart ongoing data injection — but using evolved schema.

In [ ]:
q("SELECT * FROM isk.hotset.transactions ORDER BY kafka_offset DESC LIMIT 15")

Right now our data is spread across Kafka and Iceberg. The Iceberg share (coldset) remains static in size but the Kafka share (hotset) is continuously growing.

Unbounded growth like this is often not optimal. Streambased allows you to use your Iceberg engine (in this case Spark) to easily transfer data from hotset to coldset.

Let's start by looking at the current data population in each set for the transactions table.

In [ ]:
show_distribution()

Moving data from Kafka to Iceberg is a simple `INSERT INTO ... SELECT FROM` statement with Streambased — we simply select from the hotset into the coldset.

Transferring data in this way has the following advantages:

* Transfer from hotset to coldset is not a pre-requisite for data access
* It can be scheduled during low resource usage periods.
* Data can be accumulated in Kafka until it is efficient to transfer it (no small files).
* The transfer process is atomic and involves no downtime.

But first we need to add the new column to the coldset — reflecting the Kafka schema evolution.

In [ ]:
q("ALTER TABLE direct.coldset.transactions ADD COLUMN FraudRiskScore double AFTER TransactionTime")

In [ ]:
q("DESCRIBE direct.coldset.transactions")

And finally move the data using the `INSERT INTO ... SELECT FROM` statement.

In [ ]:
q("INSERT INTO direct.coldset.transactions SELECT * FROM isk.hotset.transactions")

After transfer the coldset population has increased. Streambased will now serve a larger share of the merged dataset from the coldset.

Why hasn't the hotset population decreased?

Streambased will not explicitly delete data from Kafka. Kafka deletion is handled by the topic retention policy as usual.

The merged set now contains all of the coldset + the section of hotset that has not yet been transferred to the coldset, as you can see below.

In [ ]:
show_distribution()

The hot/cold lifecycle is complete. Data originated in Kafka (hotset), was seamlessly unified with existing Iceberg data in the merged view, and has now been durably archived to the coldset — all without any downtime or schema lock-in.

From here you can explore further: run time-travel queries against the coldset snapshots, inspect Kafka lag independently of query latency, or try the advanced demos covering KSI-accelerated lookups and Slipstream change feeds.